# Experiment 5: Learned Query Router (replaces Experiment 4's hand-crafted rules)

Experiment 4 showed a large oracle gap (86.7% vs. 76.1% for the best fixed strategy)
that a hand-crafted rule-based router failed to capture (48.3% classification
accuracy, statistically indistinguishable from doing nothing). This experiment tests
whether a **learned classifier** can close that gap.

**Methodological requirement**: with only 180 labeled queries, all routing decisions
are produced via **5-fold stratified cross-validation** — every query's predicted
route comes from a model that never saw that query during training. No number in this
notebook is computed by evaluating a model on its own training data.

Self-contained: re-loads `docs.jsonl` and `qa_final_180.jsonl` and rebuilds
Dense/BM25/Hybrid/Rerank from scratch.

In [1]:
!pip install -q sentence-transformers faiss-cpu rank_bm25 scipy scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 26.5 MB/s eta 0:00:00


## Step 1: Upload data

In [3]:
from google.colab import drive
import os, json, re
import numpy as np

# Mount Google Drive
drive.mount('/content/drive')

# Your RAG Experiment 5 folder
DATA_DIR = '/content/drive/MyDrive/AI_Project/RAG Research/Exp5/data'

# Check files
print("Files in Drive folder:")
print(os.listdir(DATA_DIR))

# Load input files directly from Google Drive
docs = [
    json.loads(l)
    for l in open(os.path.join(DATA_DIR, 'docs.jsonl'), encoding='utf-8')
]

qa = [
    json.loads(l)
    for l in open(os.path.join(DATA_DIR, 'qa_final_180.jsonl'), encoding='utf-8')
]

texts = [d['text'] for d in docs]
ids = [d['id'] for d in docs]

print(f"{len(docs)} docs, {len(qa)} questions")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Files in Drive folder:
['qa_pilot_60.jsonl', 'qa_final_180.jsonl', 'docs.jsonl']
60 docs, 180 questions


## Step 2: Rebuild Dense, BM25, Hybrid, Rerank (identical to Experiments 2-4)

In [4]:
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

model = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = model.encode(texts, convert_to_numpy=True, show_progress_bar=True)
index = faiss.IndexFlatL2(doc_embeddings.shape[1])
index.add(doc_embeddings)

tokenized_corpus = [t.lower().split() for t in texts]
bm25 = BM25Okapi(tokenized_corpus)
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
doc_text_by_id = {d['id']: d['text'] for d in docs}

def _norm(x):
    rng = x.max() - x.min()
    return (x - x.min()) / (rng + 1e-9)

ALPHA = 0.75
COST = {"dense": 1.0, "hybrid": 1.1, "hybrid_rerank": 1.1 + 10*2.0}

def dense_scores(query):
    q_emb = model.encode([query])
    return -np.linalg.norm(doc_embeddings - q_emb, axis=1), q_emb

def rank_dense(query, k):
    scores, _ = dense_scores(query)
    top = np.argsort(scores)[::-1][:k]
    return [ids[i] for i in top]

def rank_hybrid(query, k, alpha=ALPHA):
    bm25_scores = np.array(bm25.get_scores(query.lower().split()))
    vec_scores, _ = dense_scores(query)
    combined = alpha * _norm(vec_scores) + (1 - alpha) * _norm(bm25_scores)
    top = np.argsort(combined)[::-1][:k]
    return [ids[i] for i in top]

def rank_hybrid_rerank(query, k, shortlist_size=10):
    shortlist = rank_hybrid(query, shortlist_size)
    pairs = [(query, doc_text_by_id[d]) for d in shortlist]
    scores = reranker.predict(pairs)
    order = np.argsort(scores)[::-1]
    return [shortlist[i] for i in order][:k]

METHODS = {"dense": rank_dense, "hybrid": rank_hybrid, "hybrid_rerank": rank_hybrid_rerank}

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

## Step 3: Precompute ranked lists + oracle labels + router features for every query

In [5]:
def hit_at_k(ranked, target, k): return int(target in ranked[:k])
def rr(ranked, target): return 1.0/(ranked.index(target)+1) if target in ranked else 0.0

results_by_method = {m: [] for m in METHODS}
for item in qa:
    for name, fn in METHODS.items():
        results_by_method[name].append(fn(item['question'], 5))

targets = [item['expected_doc_id'] for item in qa]
true_categories = [item['category'] for item in qa]
n = len(qa)

# Oracle label per query: which method achieves the best (highest) reciprocal rank for this query
method_names = list(METHODS.keys())
oracle_label = []
for i in range(n):
    rrs = {m: rr(results_by_method[m][i], targets[i]) for m in method_names}
    oracle_label.append(max(rrs, key=rrs.get))

from collections import Counter
print("Oracle label distribution:", Counter(oracle_label))

Oracle label distribution: Counter({'dense': 141, 'hybrid_rerank': 21, 'hybrid': 18})


## Step 4: Feature engineering (cheap, available at inference time, no label leakage)

- `n_tokens`: query length
- `has_status_code`: regex for 4xx/5xx
- `has_underscore_term`: regex for `word_word` tokens
- `bm25_top`, `dense_top`: top-1 raw scores from each retriever (normalized per-query)
- `score_gap`: dense_top_norm - bm25_top_norm (a rough agreement/confidence signal)

In [6]:
def featurize(query):
    toks = query.lower().split()
    has_code = float(bool(re.search(r'\b[45]\d\d\b', query)))
    has_us = float(bool(re.search(r'[a-z]+_[a-z]+', query.lower())))
    bm25_scores = np.array(bm25.get_scores(toks))
    vec_scores, _ = dense_scores(query)
    bm25_n = _norm(bm25_scores); dense_n = _norm(vec_scores)
    return {
        "n_tokens": len(toks),
        "has_status_code": has_code,
        "has_underscore_term": has_us,
        "bm25_top": float(bm25_n.max()),
        "dense_top": float(dense_n.max()),
        "score_gap": float(dense_n.max() - bm25_n.max()),
    }

import pandas as pd
feat_rows = [featurize(item['question']) for item in qa]
X = pd.DataFrame(feat_rows)
X.head()

,n_tokens,has_status_code,has_underscore_term,bm25_top,dense_top,score_gap
0,14,0.0,1.0,1.0,1.0,9.299295e-11
1,23,0.0,0.0,1.0,1.0,2.765692e-10
2,34,0.0,0.0,1.0,1.0,8.550616e-11
3,12,1.0,0.0,1.0,1.0,9.381140e-11
4,19,0.0,0.0,1.0,1.0,1.599857e-10


## Step 5: 5-fold stratified cross-validation — two learned router variants

Both variants use out-of-fold predictions only (a query's route is always predicted by
a model that did not see that query in training). Stratified by the *true category*
label so each fold has a balanced mix of keyword/semantic/troubleshooting queries.

In [7]:
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_category = np.array(true_categories)
y_oracle_method = np.array(oracle_label)

oof_category_pred = np.empty(n, dtype=object)
oof_method_pred_direct = np.empty(n, dtype=object)

for train_idx, test_idx in skf.split(X, y_category):
    scaler = StandardScaler().fit(X.iloc[train_idx])
    Xtr, Xte = scaler.transform(X.iloc[train_idx]), scaler.transform(X.iloc[test_idx])

    # Variant A: predict category, then map to method via the same fixed mapping as Experiment 4
    clf_cat = LogisticRegression(max_iter=1000, multi_class='multinomial').fit(Xtr, y_category[train_idx])
    oof_category_pred[test_idx] = clf_cat.predict(Xte)

    # Variant B: predict the oracle-best method directly (skips the category proxy)
    clf_method = LogisticRegression(max_iter=1000, multi_class='multinomial').fit(Xtr, y_oracle_method[train_idx])
    oof_method_pred_direct[test_idx] = clf_method.predict(Xte)

ROUTE_METHOD = {'keyword': 'dense', 'troubleshooting': 'hybrid', 'semantic': 'hybrid_rerank'}
cat_router_method = [ROUTE_METHOD[c] for c in oof_category_pred]

print("Category classifier CV accuracy (vs true category):", (oof_category_pred==y_category).mean())
print("Direct method classifier CV accuracy (vs oracle label):", (oof_method_pred_direct==y_oracle_method).mean())

Category classifier CV accuracy (vs true category): 0.9388888888888889
Direct method classifier CV accuracy (vs oracle label): 0.7833333333333333


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and wi

## Step 6: Evaluate both learned routers' actual retrieval accuracy + cost, alongside Experiment 4's rule router and the fixed baselines

In [8]:
def predict_category_rule(query):
    toks = query.lower().split()
    has_code = bool(re.search(r'\b[45]\d\d\b', query)) or bool(re.search(r'[a-z]+_[a-z]+', query.lower()))
    if has_code and len(toks) < 12: return 'keyword'
    if len(toks) >= 14: return 'troubleshooting'
    return 'semantic'

rule_router_method = [ROUTE_METHOD[predict_category_rule(item['question'])] for item in qa]

def eval_routing(method_choices):
    h1,h3,h5,rrs,costs = [],[],[],[],[]
    for i,m in enumerate(method_choices):
        r = results_by_method[m][i]; t = targets[i]
        h1.append(hit_at_k(r,t,1)); h3.append(hit_at_k(r,t,3)); h5.append(hit_at_k(r,t,5))
        rrs.append(rr(r,t)); costs.append(COST[m])
    return {"recall@1": np.mean(h1), "recall@3": np.mean(h3), "recall@5": np.mean(h5),
            "mrr": np.mean(rrs), "avg_cost": np.mean(costs), "_h1": np.array(h1)}

results_table = {
    "Always dense": eval_routing(['dense']*n),
    "Always hybrid": eval_routing(['hybrid']*n),
    "Always hybrid_rerank": eval_routing(['hybrid_rerank']*n),
    "Rule router (Exp 4)": eval_routing(rule_router_method),
    "Learned category router (CV)": eval_routing(cat_router_method),
    "Learned direct router (CV)": eval_routing(list(oof_method_pred_direct)),
}
oracle_res = eval_routing(oracle_label)
results_table["Oracle (upper bound)"] = oracle_res

summary_df = pd.DataFrame([{**{k:v for k,v in r.items() if k!='_h1'}, 'method': name}
                            for name, r in results_table.items()])[['method','recall@1','recall@3','recall@5','mrr','avg_cost']]
summary_df

,method,recall@1,recall@3,recall@5,mrr,avg_cost
0,Always dense,0.722222,0.855556,0.927778,0.801204,1.000000
1,Always hybrid,0.761111,0.911111,0.933333,0.830648,1.100000
2,Always hybrid_rerank,0.716667,0.872222,0.922222,0.800000,21.100000
3,Rule router (Exp 4),0.761111,0.911111,0.933333,0.831574,4.642778
4,Learned category router (CV),0.783333,0.916667,0.961111,0.848889,6.952778
5,Learned direct router (CV),0.727778,0.872222,0.944444,0.811389,2.228333
6,Oracle (upper bound),0.866667,0.938889,0.972222,0.907130,3.355000


## Step 7: Statistical significance — best learned router vs. Rule router (Exp 4) and vs. best fixed baseline

In [9]:
def paired_bootstrap(a, b, n_boot=10000, seed=42):
    rng = np.random.default_rng(seed); nq = len(a)
    obs = a.mean() - b.mean()
    diffs = np.array([ (a[idx].mean()-b[idx].mean()) for idx in
                        (rng.integers(0,nq,nq) for _ in range(n_boot)) ])
    lo,hi = np.percentile(diffs,[2.5,97.5])
    return {"observed_diff": obs, "ci_95": (lo,hi), "p_one_sided": float((diffs<=0).mean())}

def mcnemar(a,b):
    arb=int(((a==1)&(b==0)).sum()); awb=int(((a==0)&(b==1)).sum()); nd=arb+awb
    if nd==0: return {"a_right_b_wrong":0,"a_wrong_b_right":0,"chi2":0.0,"p_value":1.0}
    chi2=(abs(arb-awb)-1)**2/nd
    from scipy.stats import chi2 as chi2_dist
    return {"a_right_b_wrong":arb,"a_wrong_b_right":awb,"chi2":chi2,"p_value":1-chi2_dist.cdf(chi2,df=1)}

best_fixed_name = max(['Always dense','Always hybrid','Always hybrid_rerank'], key=lambda k: results_table[k]['recall@1'])
best_learned_name = max(['Learned category router (CV)','Learned direct router (CV)'], key=lambda k: results_table[k]['recall@1'])
print(f"Best fixed: {best_fixed_name} | Best learned router: {best_learned_name}")

a = results_table[best_learned_name]['_h1']
print(f"\n=== {best_learned_name} vs {best_fixed_name} ===")
print("Bootstrap:", paired_bootstrap(a, results_table[best_fixed_name]['_h1']))
print("McNemar:  ", mcnemar(a, results_table[best_fixed_name]['_h1']))

print(f"\n=== {best_learned_name} vs Rule router (Exp 4) ===")
print("Bootstrap:", paired_bootstrap(a, results_table['Rule router (Exp 4)']['_h1']))
print("McNemar:  ", mcnemar(a, results_table['Rule router (Exp 4)']['_h1']))

Best fixed: Always hybrid | Best learned router: Learned category router (CV)

=== Learned category router (CV) vs Always hybrid ===
Bootstrap: {'observed_diff': np.float64(0.022222222222222254), 'ci_95': (np.float64(-0.01666666666666672), np.float64(0.06666666666666665)), 'p_one_sided': 0.1696}
McNemar:   {'a_right_b_wrong': 9, 'a_wrong_b_right': 5, 'chi2': 0.6428571428571429, 'p_value': np.float64(0.4226780741706353)}

=== Learned category router (CV) vs Rule router (Exp 4) ===
Bootstrap: {'observed_diff': np.float64(0.022222222222222254), 'ci_95': (np.float64(-0.01666666666666672), np.float64(0.06666666666666665)), 'p_one_sided': 0.1701}
McNemar:   {'a_right_b_wrong': 9, 'a_wrong_b_right': 5, 'chi2': 0.6428571428571429, 'p_value': np.float64(0.4226780741706353)}


## Step 8: Feature importance (direct-method classifier, refit on all data for interpretation only — not used for any reported accuracy number)

In [10]:
scaler_full = StandardScaler().fit(X)
clf_full = LogisticRegression(max_iter=1000, multi_class='multinomial').fit(scaler_full.transform(X), y_oracle_method)
importance_df = pd.DataFrame(clf_full.coef_, columns=X.columns, index=clf_full.classes_)
print("Logistic regression coefficients (magnitude = feature importance per class):")
importance_df

Logistic regression coefficients (magnitude = feature importance per class):


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


,n_tokens,has_status_code,has_underscore_term,bm25_top,dense_top,score_gap
dense,-0.535513,0.082419,0.328891,0.334577,0.0,-0.334578
hybrid,0.475111,-0.336149,-0.413721,-0.107141,0.0,0.107141
hybrid_rerank,0.060402,0.253730,0.084830,-0.227436,0.0,0.227437


## Step 9: Cost-accuracy Pareto sweep

Instead of a hard classification, sweep a probability threshold on the direct
classifier's predicted probability for `hybrid_rerank`: only escalate to the expensive
reranker when the classifier is sufficiently confident it's needed, else fall back to
plain hybrid. This traces out a cost-accuracy tradeoff curve rather than one operating
point.

In [11]:
from sklearn.linear_model import LogisticRegression as LR2
oof_proba_rerank = np.zeros(n)
y_binary = (y_oracle_method == 'hybrid_rerank').astype(int)
for train_idx, test_idx in skf.split(X, y_category):
    scaler = StandardScaler().fit(X.iloc[train_idx])
    clf_bin = LR2(max_iter=1000).fit(scaler.transform(X.iloc[train_idx]), y_binary[train_idx])
    oof_proba_rerank[test_idx] = clf_bin.predict_proba(scaler.transform(X.iloc[test_idx]))[:,1]

pareto_rows = []
for thresh in [0.0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.01]:
    choices = ['hybrid_rerank' if p >= thresh else 'hybrid' for p in oof_proba_rerank]
    res = eval_routing(choices)
    pareto_rows.append({"threshold": thresh, "recall@1": res['recall@1'], "avg_cost": res['avg_cost']})
pareto_df = pd.DataFrame(pareto_rows)
pareto_df

,threshold,recall@1,avg_cost
0,0.00,0.716667,21.100000
1,0.20,0.766667,3.322222
2,0.40,0.766667,1.877778
3,0.50,0.761111,1.433333
4,0.60,0.766667,1.322222
5,0.80,0.766667,1.211111
6,1.01,0.761111,1.100000


## Step 10: Save everything

In [12]:
exp5_export = {
    "summary_table": {
        name: {k: v for k, v in r.items() if k != '_h1'}
        for name, r in results_table.items()
    },

    "category_classifier_cv_accuracy": float(
        (oof_category_pred == y_category).mean()
    ),

    "direct_classifier_cv_accuracy": float(
        (oof_method_pred_direct == y_oracle_method).mean()
    ),

    "best_fixed_baseline": best_fixed_name,

    "best_learned_router": best_learned_name,

    "significance_vs_best_fixed": {
        "bootstrap": paired_bootstrap(
            results_table[best_learned_name]['_h1'],
            results_table[best_fixed_name]['_h1']
        ),
        "mcnemar": mcnemar(
            results_table[best_learned_name]['_h1'],
            results_table[best_fixed_name]['_h1']
        )
    },

    "significance_vs_rule_router": {
        "bootstrap": paired_bootstrap(
            results_table[best_learned_name]['_h1'],
            results_table['Rule router (Exp 4)']['_h1']
        ),
        "mcnemar": mcnemar(
            results_table[best_learned_name]['_h1'],
            results_table['Rule router (Exp 4)']['_h1']
        )
    },
}

# Save ALL Experiment 5 outputs directly to Google Drive
with open(
    os.path.join(DATA_DIR, 'experiment5_results.json'),
    'w',
    encoding='utf-8'
) as f:
    json.dump(exp5_export, f, indent=2, default=float)

summary_df.to_csv(
    os.path.join(DATA_DIR, 'experiment5_summary_table.csv'),
    index=False
)

importance_df.to_csv(
    os.path.join(DATA_DIR, 'experiment5_feature_importance.csv'),
    index=False
)

pareto_df.to_csv(
    os.path.join(DATA_DIR, 'experiment5_pareto_sweep.csv'),
    index=False
)

print("Saved to Google Drive:")
print("experiment5_results.json")
print("experiment5_summary_table.csv")
print("experiment5_feature_importance.csv")
print("experiment5_pareto_sweep.csv")

Saved to Google Drive:
experiment5_results.json
experiment5_summary_table.csv
experiment5_feature_importance.csv
experiment5_pareto_sweep.csv


## What to report in the paper
- **Summary table** (Step 6): does either learned router close the oracle gap more than Experiment 4's rule router? Report honestly if not.
- **CV accuracy** (Step 5) vs. Experiment 4's 48.3% rule-based classification accuracy — direct comparison of learned vs. hand-crafted.
- **Significance tests** (Step 7): learned router vs. best fixed strategy, and vs. the Experiment 4 rule router specifically.
- **Feature importance** (Step 8): which signals (length, regex, retrieval-score-gap) actually drive the learned router's decisions — this is what a hand-crafted rule-writer would need to know.
- **Pareto curve** (Step 9): the cost-accuracy tradeoff as the escalation threshold varies, rather than a single operating point — this is the strongest "systems contribution" framing for an IEEE-style venue.
- If the learned router still falls well short of the oracle, or isn't significantly better than Experiment 4's rule router, report that plainly — with only 180 examples and 6 features, a modest classifier may simply not have enough signal, which is itself a useful conclusion about what would be needed (more training data, better features) to close the gap.